# Session 2: the SIR equations
**A. Real-world problem.** A town wants to know two things before any outbreak: how many
people would catch a new infection in total, and what share of the town must be protected
(by vaccination, say) so the infection cannot spread. Both answers come from three
equations and one number, R0.

**Important.** Teaching model only: not a forecast, not medical advice, not a basis for
public-health decisions.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the equations
With N people, S + I + R = N always, and

    dS/dt = -beta S I / N
    dI/dt =  beta S I / N - gamma I
    dR/dt =  gamma I

beta [1/day] = infectious contacts per person per day, gamma [1/day] = recovery rate
(1/gamma = days infectious). **R0 = beta / gamma.** The infection grows exactly when
dI/dt > 0, i.e. when R0 * S / N > 1.

In [ ]:
def r0(beta, gamma):
    return beta / gamma


def r_effective(R0, s_fraction):
    """Reproduction number when a fraction s of people is still susceptible."""
    return R0 * s_fraction


def herd_immunity_threshold(R0):
    """Share of people who must be immune so that R0 * (1 - share) < 1."""
    return max(0.0, 1.0 - 1.0 / R0)


def simulate(beta, gamma, N=10000, I0=10, days=160, R_init=0.0, dt=0.05):
    """Euler steps of size dt (days). Returns S, I, R, one value per day."""
    S, I, R = N - I0 - R_init, float(I0), float(R_init)
    out = [(S, I, R)]
    per_day = int(round(1 / dt))
    for _ in range(days):
        for _ in range(per_day):
            flow = beta * S * I / N * dt
            rec = gamma * I * dt
            S, I, R = S - flow, I + flow - rec, R + rec
        out.append((S, I, R))
    return np.array(out).T

## Step 2: the final size
When the outbreak is over, the fraction z of the town ever infected obeys

    z = 1 - exp(-R0 z)

(derived by dividing dS/dt by dR/dt; assumes almost everyone starts susceptible). There is
no closed formula, so we solve it numerically with Newton's method.

In [ ]:
def final_size(R0, z0=0.9):
    """Solve z = 1 - exp(-R0 z) by Newton's method; returns 0 if R0 <= 1."""
    if R0 <= 1.0:
        return 0.0
    z = z0
    for _ in range(100):
        f = z - (1 - np.exp(-R0 * z))
        df = 1 - R0 * np.exp(-R0 * z)
        z_new = z - f / df
        if abs(z_new - z) < 1e-13:
            return float(z_new)
        z = z_new
    return float(z)


def peak_fraction(R0):
    """Largest fraction infected at once, when almost everyone starts susceptible: 1 - (1 + ln R0) / R0."""
    return 1.0 - (1.0 + np.log(R0)) / R0 if R0 > 1 else 0.0

## Step 3: design with vaccination
A vaccine with efficacy e protects a fraction e of the people vaccinated (the rest stay
susceptible, which is what "leaky" or imperfect protection means for the town as a whole).
With coverage v, the protected share is v * e, and we need R0 * (1 - v * e) < 1:

    v > (1 - 1/R0) / e

In [ ]:
def protected_share(coverage, efficacy):
    return coverage * efficacy


def coverage_needed(R0, efficacy=1.0):
    """Smallest coverage that brings R_eff to 1. A value above 1 means: impossible by vaccination alone."""
    return herd_immunity_threshold(R0) / efficacy

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("R0 = 3 herd threshold:", round(herd_immunity_threshold(3.0), 4))
    print("final size R0 = 2:", round(final_size(2.0), 4), " R0 = 3:", round(final_size(3.0), 4))
    print("peak fraction R0 = 2:", round(peak_fraction(2.0), 4))
    print("coverage needed R0 = 3, e = 1.0 / 0.9 / 0.6:", [round(coverage_needed(3.0, e), 3) for e in (1.0, 0.9, 0.6)])

    gamma = 0.2
    S, I, R = simulate(0.4, gamma)
    print("simulated peak fraction (R0 = 2):", round(I.max() / 10000, 4), " final R:", round(R[-1] / 10000, 4))

    d = np.arange(len(S))
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for v, name in ((0.0, "no vaccination"), (0.3, "30% covered, e = 0.9"), (0.5, "50% covered, e = 0.9")):
        _, Iv, _ = simulate(0.4, gamma, R_init=protected_share(v, 0.9) * 10000)
        ax.plot(d, Iv, label=name)
        print(name, "peak infected:", round(Iv.max(), 1))
    ax.set_xlabel("day")
    ax.set_ylabel("people infected now (out of 10000)")
    ax.set_title("R0 = 2: more protection, lower and later peak")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "vaccination_curves.png", dpi=120)

    r0s = np.linspace(1.0, 5.0, 81)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.plot(r0s, [100 * final_size(x) for x in r0s], label="ever infected (final size)")
    ax2.plot(r0s, [100 * herd_immunity_threshold(x) for x in r0s], "--", label="herd immunity threshold")
    ax2.set_xlabel("R0 (no unit)")
    ax2.set_ylabel("percent of the town")
    ax2.set_title("Final size and herd immunity threshold versus R0")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "final_size.png", dpi=120)

## Where this stops
Real vaccines wane, people mix unevenly, and the formulas assume a well-mixed town. The
thresholds here are idealised teaching numbers, never targets for a real programme.

## Coding tasks
1. **Easy.** Use `final_size` to print the final size for R0 = 1.5, 2, 3 and 4. At which R0 does it first pass 90 percent?
2. **Stretch.** Use `simulate` to find the coverage (efficacy 0.9, whole percent) at which the peak first falls below 100 people for R0 = 2. Compare with the threshold.
3. **Explain.** Explain why the final size is larger than the herd immunity threshold, using the words overshoot and peak.

## Assessment
- **A. Conceptual.** Explain in words why the herd immunity threshold is 1 - 1/R0 and why a larger R0 needs a larger share protected.
- **B. Numerical.** R0 = 4. Find the herd immunity threshold. With a vaccine of efficacy 0.8, what coverage is needed?
- **C. Graph and simulation interpretation.** In the vaccination figure, how do the peak height and peak day change as coverage goes from 0 to 30 to 50 percent? What does this tell you about R_eff?
- **D. Design and optimisation challenge.** Design: for R0 = 3 the town can only buy a vaccine with efficacy 0.6. Is vaccination alone enough to stop the infection spreading? What combination of measures could work, and what assumption does your answer rely on?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("R0 = beta / gamma = 2 for 0.4 and 0.2", abs(L.r0(0.4, 0.2) - 2.0) < 1e-12))
    checks.append(("worked example: herd immunity threshold for R0 = 3 is 66.7 percent",
                   abs(L.herd_immunity_threshold(3.0) - 2 / 3) < 1e-12))
    z = L.final_size(2.0)
    checks.append((f"worked example: final size for R0 = 2 is 79.7 percent (got {100 * z:.2f})", abs(100 * z - 79.68) < 0.01))
    checks.append(("the final size really solves z = 1 - exp(-R0 z)", abs(z - (1 - np.exp(-2.0 * z))) < 1e-12))
    checks.append(("final size for R0 = 3 is 94.0 percent", abs(100 * L.final_size(3.0) - 94.05) < 0.01))
    checks.append(("edge: R0 <= 1 gives final size 0 and threshold 0",
                   L.final_size(1.0) == 0.0 and L.final_size(0.5) == 0.0 and L.herd_immunity_threshold(0.8) == 0.0))
    checks.append(("peak fraction formula for R0 = 2 is 15.3 percent", abs(100 * L.peak_fraction(2.0) - 15.34) < 0.01))

    S, I, R = L.simulate(0.4, 0.2)
    N = 10000
    checks.append(("S + I + R = N on every day", float(np.abs(S + I + R - N).max()) < 1e-6))
    checks.append(("simulated peak is within 1 percent of N of the formula (15.4 vs 15.3 percent)",
                   abs(I.max() / N - L.peak_fraction(2.0)) < 0.01 and abs(100 * I.max() / N - 15.4) < 0.05))
    checks.append(("simulated final size matches the equation within 0.2 percent of N (79.8 vs 79.7)",
                   abs(R[-1] / N - z) < 0.002 and abs(100 * R[-1] / N - 79.8) < 0.05))

    # growth rule: grows when R0 * S / N > 1
    checks.append(("I grows at the start when R0 S/N > 1 and falls when R0 S/N < 1",
                   (np.diff(I)[0] > 0) and L.r_effective(2.0, 0.6) > 1 and L.r_effective(2.0, 0.4) < 1))
    S2, I2, R2 = L.simulate(0.4, 0.2, R_init=0.6 * N)
    checks.append(("with 60 percent protected (R_eff = 0.8) the outbreak shrinks from day 0", bool(I2.max() == I2[0])))

    # design problem
    checks.append(("coverage needed for R0 = 3 and efficacy 0.9 is 74.1 percent", abs(100 * L.coverage_needed(3.0, 0.9) - 74.07) < 0.01))
    checks.append(("efficacy 0.6 and R0 = 3 needs more than 100 percent coverage: impossible", L.coverage_needed(3.0, 0.6) > 1.0))
    c = L.coverage_needed(3.0, 0.9)
    checks.append(("at that coverage R_eff is exactly 1", abs(L.r_effective(3.0, 1 - L.protected_share(c, 0.9)) - 1.0) < 1e-12))
    p30 = simulate_peak(L, 0.3)
    p50 = simulate_peak(L, 0.5)
    checks.append((f"R0 = 2 peaks: none 1542.9, 30 percent covered 415.2, 50 percent covered 32.6 people (got {p30:.1f}, {p50:.1f})",
                   abs(L.simulate(0.4, 0.2)[1].max() - 1542.9) < 0.1 and abs(p30 - 415.2) < 0.1 and abs(p50 - 32.6) < 0.1))
    checks.append(("worked example: z = 0.797 satisfies z = 1 - exp(-2 z) to 0.001 (1 - exp(-1.594) = 0.797)", abs(1 - np.exp(-2 * 0.797) - 0.797) < 1e-3))
    checks.append(("R_eff at the start is 2, 1.46 and 1.10 for 0, 30, 50 percent coverage (efficacy 0.9)",
                   [round(L.r_effective(2.0, 1 - L.protected_share(c, 0.9)), 2) for c in (0.0, 0.3, 0.5)] == [2.0, 1.46, 1.1]))
    checks.append(("R0 = 4: threshold 75 percent; efficacy 0.8 needs 93.75 percent", abs(L.herd_immunity_threshold(4.0) - 0.75) < 1e-12 and abs(L.coverage_needed(4.0, 0.8) - 0.9375) < 1e-12))
    checks.append(("R0 = 3, efficacy 0.6: full coverage leaves R_eff = 1.2; with R0 = 2.5 full coverage just reaches the threshold",
                   abs(L.r_effective(3.0, 1 - L.protected_share(1.0, 0.6)) - 1.2) < 1e-12 and abs(L.coverage_needed(2.5, 0.6) - 1.0) < 1e-12))
    checks.append(("coverage 0.741 at efficacy 0.9 protects 66.7 percent", abs(L.protected_share(0.741, 0.9) - 0.667) < 1e-3))
    return checks


def simulate_peak(L, coverage, efficacy=0.9):
    return float(L.simulate(0.4, 0.2, R_init=L.protected_share(coverage, efficacy) * 10000)[1].max())

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
